In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

In [ ]:
class LabPics:
    def __init__(self, main_path):
        image_path = os.path.join(main_path, "Image")
        mask_path = os.path.join(main_path, "Instance")
        imgs, masks = [], []
        # TODO: Populate imgs and masks lists
        self.paths = {"image": imgs, "masks": masks}
    
    def __len__(self):
        return len(self.paths["image"])
    
    def __getitem__(self, index):
        # TODO: Implement __getitem__ method
        pass

In [ ]:
output = dataset[129]
img, mask, points, labels = output["img"], output["mask"], output["points"], output["labels"]

fig, axes = plt.subplots(1, 3, figsize=(10, 10))

# Original image
axes[0].imshow(img)
axes[0].set_title('Original Image')

# Segmentation mask with background
bg_mask = (mask.sum(axis=0) == 0).astype(float)
new_mask = np.vstack([bg_mask[np.newaxis, :, :], mask])
axes[1].imshow(new_mask.argmax(0))
axes[1].set_title('Segmentation Mask')

# Overlay with points
colorful = np.zeros_like(img, dtype=float)
for ind in range(mask.shape[0]):
    axes[2].scatter(points[ind][0][0], points[ind][0][1], c='red', s=50)
    colorful += mask[ind][:, :, np.newaxis] * np.random.randint(0, 255, size=3)

axes[2].imshow((0.5 * colorful + 0.5 * img).astype(np.uint8))
axes[2].set_title('Overlay with Points')

plt.tight_layout()
plt.show()


In [ ]:
checkpoint = ...
model_cfg = ...
predictor = SAM2ImagePredictor(build_sam2(model_cfg, checkpoint))

In [ ]:
scaler = torch.amp.GradScaler(device="cuda")
predictor.model.sam_prompt_encoder.train(True)
predictor.model.sam_mask_decoder.train(True)
optimizer = torch.optim.AdamW(predictor.model.parameters(), lr=1e-4, weight_decay=5e-4)

In [ ]:
class AverageMeter:
    """
    Computes and stores the average and current value.
    """
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count if self.count > 0 else 0

In [ ]:
accumulation_step = 4
import tqdm
dataloader = ...
ScoreLoss = AverageMeter()
CELoss = AverageMeter()
for epoch in range(10):
    with tqdm.tqdm(dataloader, desc=f"Epoch {epoch+1}/10") as pbar:
        for step, output in enumerate(pbar):
            # TODO: train step
            pbar.set_postfix({"CE Loss": f'{CELoss.avg:.4f}', "IoU": f'{ScoreLoss.avg:.4f}'})
            if step % 100 == 0 and step > 0:
                CELoss.reset()
                ScoreLoss.reset()

In [ ]:
# Evaluation with visualization
import matplotlib.pyplot as plt

predictor.model.eval()
test_idx = np.random.randint(len(test_dataset))
output = test_dataset[test_idx]
img, mask_gt, points, labels = output["img"], output["mask"], output["points"], output["labels"]

with torch.no_grad():
    predictor.set_image(img)
    mask_input, unnorm_coords, labels_prep, unnorm_box = predictor._prep_prompts(
        points, labels, None, None, True
    )
    masks, iou_predictions, _ = predictor._predict(
        unnorm_coords,
        labels_prep,
        unnorm_box,
        mask_input,
        True,
        return_logits=False,
    )
    # Move to CPU and convert to numpy
    masks = masks.cpu().numpy()
    iou_predictions = iou_predictions.cpu().numpy()
    # Take the first mask output (best mask) for each object
    masks = masks[:, 0, :, :]  # Shape: (num_objects, H, W)

# Visualize results
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Original image with points
axes[0].imshow(img)
for point in points:
    axes[0].scatter(point[0][0], point[0][1], c='red', s=100, marker='*')
axes[0].set_title('Input Image with Points')
axes[0].axis('off')

# Ground truth masks
gt_viz = np.zeros_like(img, dtype=float)
for i, mask in enumerate(mask_gt):
    color = np.random.randint(50, 255, size=3)
    gt_viz += mask[:, :, None] * color
axes[1].imshow(img)
axes[1].imshow(gt_viz.astype(np.uint8), alpha=0.5)
axes[1].set_title('Ground Truth')
axes[1].axis('off')

# Predicted masks
pred_viz = np.zeros_like(img, dtype=float)
for i, mask in enumerate(masks):
    color = np.random.randint(50, 255, size=3)
    pred_viz += mask[:, :, None] * color
axes[2].imshow(img)
axes[2].imshow(pred_viz.astype(np.uint8), alpha=0.5)
axes[2].set_title('Predictions')
axes[2].axis('off')

# IoU scores
axes[3].bar(range(len(iou_predictions)), iou_predictions[:, 0])
axes[3].set_xlabel('Mask Index')
axes[3].set_ylabel('IoU Score')
axes[3].set_title('Predicted IoU Scores')
axes[3].set_ylim([0, 1])

plt.tight_layout()
plt.show()

print(f"Mean IoU Score: {iou_predictions[:, 0].mean():.3f}")

In [ ]:
# Calculate overall IoU on entire dataset
predictor.model.eval()
all_ious = []

with torch.no_grad():
    for idx in range(len(test_dataset)):
        output = test_dataset[idx]
        img, mask_gt, points, labels = output["img"], output["mask"], output["points"], output["labels"]
        
        if len(mask_gt) == 0:
            continue
            
        predictor.set_image(img)
        mask_input, unnorm_coords, labels_prep, unnorm_box = predictor._prep_prompts(
            points, labels, None, None, True
        )
        masks, _, _ = predictor._predict(
            unnorm_coords,
            labels_prep,
            unnorm_box,
            mask_input,
            True,
            return_logits=False,
        )
        
        # Get best mask for each object
        mask_pred = masks[:, 0, :, :]  # Shape: (num_objects, H, W)
        mask_pred = (mask_pred > 0.5).cpu().numpy()
        
        # Calculate IoU for each object
        for i in range(len(mask_gt)):
            inter = (mask_pred[i] * mask_gt[i]).sum()
            union = mask_pred[i].sum() + mask_gt[i].sum() - inter
            if union > 0:
                iou = inter / union
                all_ious.append(iou)

overall_iou = np.mean(all_ious)
print(f"Overall IoU on {len(dataset)} images: {overall_iou:.4f}")
print(f"Total masks evaluated: {len(all_ious)}")